<a href="https://colab.research.google.com/github/pyshka501/rl-textbook/blob/main/translations/ru/notebooks/ch09_actor_critic_ppo_ru.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background: linear-gradient(135deg, #200122 0%, #6f0000 100%); 
padding: 40px; border-radius: 12px; color: white; margin-bottom: 20px;">
<h1 style="font-size:2.2em; margin:0 0 10px 0;">Глава 9. Актёр-критик и PPO</h1>
<p style="font-size:1.1em; opacity:0.85; margin:0;">
От A2C к Proximal Policy Optimization — GAE, отсечение и стабильное on-policy обучение на CartPole и LunarLander.
</p>
</div>

### Цели обучения
- Реализовать A2C (Advantage Actor-Critic) с нуля
- Реализовать PPO-Clip с нуля
- Понять обобщённую оценку преимущества (GAE)
- Сравнить A2C и PPO на CartPole-v1 и LunarLander-v3
- Визуализировать механизм отсечения отношения вероятностей в PPO

<div style="background: #1e3a5f; padding: 15px; border-radius: 8px; border-left: 4px solid #4fc3f7;">
  <strong style="color: #4fc3f7;">Подготовка окружения</strong><br>
  <span style="color: #cdd9e5;">Требуется <code>gymnasium</code>. Colab / Kaggle: только CPU. CartPole ~3 минуты, LunarLander ~6 минут. Всего ~9 минут.</span>
</div>

In [ ]:
!pip install -q "gymnasium[classic-control,box2d]" torch matplotlib numpy tqdm

In [ ]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt
from tqdm.auto import trange

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


<div style="background:#1b4332;padding:16px;border-radius:8px;color:white;">
<h2 style="margin:0">1 · Общая архитектура актёр-критик</h2></div>

И A2C, и PPO используют общий backbone с двумя <<головами>>:

- **Актёр** (policy head): выдаёт логиты действий → $\pi_\theta(a|s)$
- **Критик** (value head): выдаёт скаляр $V_\theta(s)$

Общие слои дают преимущество обеим целям обучения.

In [ ]:
class ActorCriticNet(nn.Module):
    """Shared-trunk actor-critic network (3-layer MLP)."""
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        # Shared backbone
        self.backbone = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden),   nn.Tanh(),
        )
        self.actor  = nn.Linear(hidden, action_dim)  # policy head
        self.critic = nn.Linear(hidden, 1)            # value head

        # Orthogonal init (common in PPO implementations)
        for layer in self.backbone:
            if isinstance(layer, nn.Linear):
                nn.init.orthogonal_(layer.weight, gain=np.sqrt(2))
                nn.init.zeros_(layer.bias)
        nn.init.orthogonal_(self.actor.weight,  gain=0.01)
        nn.init.orthogonal_(self.critic.weight, gain=1.0)

    def forward(self, x):
        h      = self.backbone(x)
        logits = self.actor(h)
        value  = self.critic(h).squeeze(-1)
        return logits, value

    def act(self, state):
        """Sample action; return (action, log_prob, value)."""
        logits, value = self(state)
        dist   = torch.distributions.Categorical(logits=logits)
        action = dist.sample()
        return action.item(), dist.log_prob(action), value


# Quick sanity check
net = ActorCriticNet(state_dim=4, action_dim=2)
dummy = torch.zeros(1, 4)
logits, val = net(dummy)
print(f"Actor  output shape: {logits.shape}  → action logits")
print(f"Critic output shape: {val.shape}   → scalar value")
print(f"Params: {sum(p.numel() for p in net.parameters()):,}")


<div style="background:#1b4332;padding:16px;border-radius:8px;color:white;">
<h2 style="margin:0">2 · Обобщённая оценка преимущества (GAE)</h2></div>

GAE (Schulman и др., 2016) интерполирует между TD(0) (низкая дисперсия, высокое смещение)
и Монте-Карло (высокая дисперсия, низкое смещение) с параметром $\lambda \in [0, 1]$:

$$
\hat{A}_t^{\text{GAE}(\gamma,\lambda)} = \sum_{l=0}^{T-t-1} (\gamma\lambda)^l \delta_{t+l}
$$

где $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$ — TD-остаток.

- $\lambda = 0$: чистый TD(0) — $\hat{A}_t = r_t + \gamma V(s_{t+1}) - V(s_t)$
- $\lambda = 1$: отдачи Монте-Карло — $\hat{A}_t = G_t - V(s_t)$

In [ ]:
def compute_gae(rewards, values, dones, next_value, gamma=0.99, lam=0.95):
    """
    Compute GAE advantages and returns.

    Args:
        rewards  : list of rewards for the rollout
        values   : tensor of V(s_t) estimates
        dones    : list of done flags (1.0 = terminal)
        next_value: V(s_{T+1}) bootstrap value
        gamma    : discount factor
        lam      : GAE lambda
    Returns:
        advantages: tensor, normalised
        returns   : tensor (used for critic loss)
    """
    T         = len(rewards)
    adv       = torch.zeros(T, device=DEVICE)
    last_gae  = 0.0
    values_np = values.detach()

    # Append bootstrap value
    values_ext = torch.cat([values_np, torch.tensor([next_value], device=DEVICE)])

    for t in reversed(range(T)):
        mask       = 1.0 - dones[t]
        delta      = rewards[t] + gamma * values_ext[t+1] * mask - values_ext[t]
        last_gae   = delta + gamma * lam * mask * last_gae
        adv[t]     = last_gae

    returns  = adv + values_np
    adv_norm = (adv - adv.mean()) / (adv.std() + 1e-8)
    return adv_norm, returns


# ── Visualise GAE for different lambda values ─────────────────────────────────
np.random.seed(0)
T_demo = 20
demo_rewards = np.random.randn(T_demo) * 0.5 + 1.0
demo_values  = torch.zeros(T_demo, device=DEVICE)  # V=0 for demo clarity
demo_dones   = torch.zeros(T_demo, device=DEVICE)

fig, ax = plt.subplots(figsize=(11, 4))
for lam, color in [(0.0, "#e74c3c"), (0.5, "#f39c12"), (0.95, "#2ecc71"), (1.0, "#3498db")]:
    adv, _ = compute_gae(demo_rewards, demo_values, demo_dones, 0.0, gamma=0.99, lam=lam)
    ax.plot(adv.cpu().numpy(), label=f"λ = {lam}", color=color, linewidth=2, marker="o", markersize=4)

ax.set_xlabel("Timestep t")
ax.set_ylabel("GAE Advantage Estimate")
ax.set_title("GAE: Effect of λ on Advantage Estimates\n(λ=0: pure TD, λ=1: Monte Carlo)", fontweight="bold")
ax.legend()
ax.axhline(0, color="gray", linestyle="--", alpha=0.4)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("ch09_gae.png", dpi=120, bbox_inches="tight")
plt.show()


<div style="background:#1d3557;padding:16px;border-radius:8px;color:white;">
<h2 style="margin:0">3 · A2C — Advantage Actor-Critic</h2></div>

A2C собирает **$n$-шаговые роллауты**, считает GAE-преимущества и делает **один шаг градиента**.

Совмещённая функция потерь:
$$
\mathcal{L} = -\underbrace{\mathbb{E}[\hat{A}_t \log \pi_\theta(a_t|s_t)]}_{\text{актёр}} 
+ c_v \underbrace{\mathbb{E}[(G_t - V_\theta(s_t))^2]}_{\text{критик}}
- c_e \underbrace{H[\pi_\theta(\cdot|s_t)]}_{\text{бонус энтропии}}
$$

In [ ]:
def rollout(env, net, n_steps=128, device=DEVICE):
    """Collect n_steps of experience from env using net."""
    states, actions, rewards, log_probs, values, dones = [], [], [], [], [], []

    state, _ = env.reset()
    done = False

    for _ in range(n_steps):
        s_t = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            action, log_prob, value = net.act(s_t)

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        states.append(state)
        actions.append(action)
        rewards.append(reward)
        log_probs.append(log_prob.squeeze())
        values.append(value.squeeze())
        dones.append(float(done))

        state = next_state
        if done:
            state, _ = env.reset()

    # Bootstrap value from final state
    s_next = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
    with torch.no_grad():
        _, next_value = net(s_next)

    return (
        torch.tensor(np.array(states), dtype=torch.float32, device=device),
        torch.tensor(actions, dtype=torch.long, device=device),
        rewards,
        torch.stack(log_probs),
        torch.stack(values),
        torch.tensor(dones, dtype=torch.float32, device=device),
        next_value.item(),
    )


def train_a2c(
    env_name="CartPole-v1",
    n_updates=500,
    n_steps=128,
    lr=3e-4,
    gamma=0.99,
    lam=0.95,
    c_v=0.5,
    c_e=0.01,
    seed=SEED,
):
    """A2C with GAE."""
    env = gym.make(env_name)
    env.reset(seed=seed)
    state_dim  = env.observation_space.shape[0]
    action_dim = env.action_space.n

    net       = ActorCriticNet(state_dim, action_dim).to(DEVICE)
    optimizer = optim.Adam(net.parameters(), lr=lr, eps=1e-5)

    all_rewards  = []
    ep_reward    = 0.0
    ep_rewards   = []

    for update in trange(n_updates, desc=f"A2C-{env_name}", leave=False):
        states, actions, rewards, old_log_probs, values, dones, next_val = \
            rollout(env, net, n_steps=n_steps)

        advantages, returns = compute_gae(rewards, values, dones, next_val, gamma, lam)

        # Re-evaluate under current policy
        logits, new_values = net(states)
        dist       = torch.distributions.Categorical(logits=logits)
        new_lp     = dist.log_prob(actions)
        entropy    = dist.entropy().mean()

        actor_loss  = -(new_lp * advantages).mean()
        critic_loss = F.mse_loss(new_values, returns.detach())
        loss        = actor_loss + c_v * critic_loss - c_e * entropy

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), 0.5)
        optimizer.step()

        # Track episodic rewards
        for r, d in zip(rewards, dones):
            ep_reward += r
            if d:
                ep_rewards.append(ep_reward)
                ep_reward = 0.0

    env.close()
    return ep_rewards


a2c_cartpole = train_a2c("CartPole-v1",    n_updates=400)
print(f"A2C CartPole — last-50 mean: {np.mean(a2c_cartpole[-50:]):.1f}")


<div style="background:#4a1942;padding:16px;border-radius:8px;color:white;">
<h2 style="margin:0">4 · PPO-Clip с нуля</h2></div>

PPO (Schulman и др., 2017) делает **несколько шагов градиента** на одних и тех же данных роллаута,
с механизмом **отсечения отношения вероятностей**:

$$
\mathcal{L}^{\text{CLIP}}(\theta) = \mathbb{E}_t\left[
\min\left(r_t(\theta)\hat{A}_t,\; \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon)\hat{A}_t\right)
\right]
$$

где $r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}$ — отношение вероятностей.

Это предотвращает **слишком большие** обновления стратегии и даёт гарантию в духе trust-region
без вычислительных затрат TRPO.

In [ ]:
def train_ppo(
    env_name="CartPole-v1",
    n_updates=400,
    n_steps=128,
    n_epochs=4,
    batch_size=64,
    lr=3e-4,
    gamma=0.99,
    lam=0.95,
    clip_eps=0.2,
    c_v=0.5,
    c_e=0.01,
    seed=SEED,
    return_clip_data=False,
):
    """PPO-Clip with multiple epochs per rollout."""
    env = gym.make(env_name)
    env.reset(seed=seed)
    state_dim  = env.observation_space.shape[0]
    action_dim = env.action_space.n

    net       = ActorCriticNet(state_dim, action_dim).to(DEVICE)
    optimizer = optim.Adam(net.parameters(), lr=lr, eps=1e-5)

    ep_rewards  = []
    ep_reward   = 0.0
    clip_fracs  = []   # fraction of samples clipped per update
    ratios_log  = []   # raw ratios for visualisation

    for update in trange(n_updates, desc=f"PPO-{env_name}", leave=False):
        states, actions, rewards, old_log_probs, values, dones, next_val = \
            rollout(env, net, n_steps=n_steps)

        advantages, returns = compute_gae(rewards, values, dones, next_val, gamma, lam)
        advantages = advantages.detach()
        returns    = returns.detach()
        old_lp     = old_log_probs.detach()

        clip_frac_ep = []

        for _ in range(n_epochs):
            # Shuffle mini-batches
            idx = torch.randperm(n_steps)
            for start in range(0, n_steps, batch_size):
                mb = idx[start:start+batch_size]

                logits, new_values = net(states[mb])
                dist    = torch.distributions.Categorical(logits=logits)
                new_lp  = dist.log_prob(actions[mb])
                entropy = dist.entropy().mean()

                ratio   = (new_lp - old_lp[mb]).exp()  # π_new / π_old
                adv_mb  = advantages[mb]

                # PPO clip objective
                surr1   = ratio * adv_mb
                surr2   = ratio.clamp(1 - clip_eps, 1 + clip_eps) * adv_mb
                actor_l = -torch.min(surr1, surr2).mean()
                critic_l= F.mse_loss(new_values, returns[mb])
                loss    = actor_l + c_v * critic_l - c_e * entropy

                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(net.parameters(), 0.5)
                optimizer.step()

                # Track clipping fraction
                clipped = ((ratio - 1.0).abs() > clip_eps).float().mean().item()
                clip_frac_ep.append(clipped)
                if return_clip_data and update > n_updates - 5:
                    ratios_log.extend(ratio.detach().cpu().numpy().tolist())

        clip_fracs.append(np.mean(clip_frac_ep))

        for r, d in zip(rewards, dones):
            ep_reward += r
            if d:
                ep_rewards.append(ep_reward)
                ep_reward = 0.0

    env.close()
    if return_clip_data:
        return ep_rewards, clip_fracs, ratios_log
    return ep_rewards


ppo_cartpole, ppo_clip_fracs, ppo_ratios = train_ppo(
    "CartPole-v1", n_updates=400, return_clip_data=True)
print(f"PPO CartPole  — last-50 mean: {np.mean(ppo_cartpole[-50:]):.1f}")


<div style="background:#1b4332;padding:16px;border-radius:8px;color:white;">
<h2 style="margin:0">5 · LunarLander-v3</h2></div>

<div style="background: #1e3a5f; padding: 15px; border-radius: 8px; border-left: 4px solid #4fc3f7;">
  <span style="color: #cdd9e5;"><strong>Время выполнения:</strong> ~3 минуты для каждого запуска на CPU в Colab.</span>
</div>

У LunarLander 8-мерное непрерывное пространство наблюдений и 4 дискретных действия.
Чтобы корабль садился безопасно, требуется больше итераций обучения.

In [ ]:
# ~6 min total on CPU
a2c_lunar  = train_a2c("LunarLander-v3",  n_updates=1000)
ppo_lunar  = train_ppo("LunarLander-v3",  n_updates=1000)

print(f"A2C LunarLander  — last-50 mean: {np.mean(a2c_lunar[-50:]):.1f}")
print(f"PPO LunarLander  — last-50 mean: {np.mean(ppo_lunar[-50:]):.1f}")


<div style="background:#4a1942;padding:16px;border-radius:8px;color:white;">
<h2 style="margin:0">6 · Визуализации: кривые обучения и отсечение PPO</h2></div>

In [ ]:
def smooth(x, w=20):
    if len(x) < w:
        return np.array(x)
    return np.convolve(x, np.ones(w)/w, mode="valid")


fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# ── CartPole learning curves ─────────────────────────────────────────────────
ax = axes[0, 0]
ax.plot(smooth(a2c_cartpole), label="A2C", color="#3498db",  linewidth=2)
ax.plot(smooth(ppo_cartpole), label="PPO", color="#e74c3c",  linewidth=2)
ax.axhline(475, color="gray", linestyle="--", alpha=0.5, label="near-optimal")
ax.set_xlabel("Episode"); ax.set_ylabel("Reward (smoothed, w=20)")
ax.set_title("CartPole-v1: A2C vs PPO", fontweight="bold")
ax.legend(); ax.grid(alpha=0.3)

# ── LunarLander learning curves ──────────────────────────────────────────────
ax2 = axes[0, 1]
ax2.plot(smooth(a2c_lunar), label="A2C", color="#3498db", linewidth=2)
ax2.plot(smooth(ppo_lunar), label="PPO", color="#e74c3c", linewidth=2)
ax2.axhline(200, color="green", linestyle="--", alpha=0.5, label="solved (200)")
ax2.axhline(0,   color="gray",  linestyle="--", alpha=0.3)
ax2.set_xlabel("Episode"); ax2.set_ylabel("Reward (smoothed, w=20)")
ax2.set_title("LunarLander-v3: A2C vs PPO", fontweight="bold")
ax2.legend(); ax2.grid(alpha=0.3)

# ── PPO clipping fraction over training ─────────────────────────────────────
ax3 = axes[1, 0]
ax3.plot(ppo_clip_fracs, color="#f39c12", linewidth=1.5, alpha=0.6, label="raw")
if len(ppo_clip_fracs) >= 10:
    ax3.plot(smooth(ppo_clip_fracs, w=10), color="#e74c3c", linewidth=2.5, label="smoothed")
ax3.set_xlabel("PPO Update")
ax3.set_ylabel("Fraction of Samples Clipped")
ax3.set_title("PPO Clipping Fraction over Training\n(high early = large updates; decreases as policy stabilises)", fontweight="bold")
ax3.axhline(0.1, color="gray", linestyle="--", alpha=0.5, label="typical target ~10%")
ax3.legend(); ax3.grid(alpha=0.3)

# ── PPO ratio vs clipped objective visualisation ─────────────────────────────
ax4 = axes[1, 1]
ratios_np = np.array(ppo_ratios if ppo_ratios else [1.0])
clip_eps  = 0.2

# Plot theoretical curve for a positive advantage
r_range  = np.linspace(0.5, 1.8, 300)
adv_pos  = 1.0
surr1_pos = r_range * adv_pos
surr2_pos = np.clip(r_range, 1-clip_eps, 1+clip_eps) * adv_pos
obj_pos   = np.minimum(surr1_pos, surr2_pos)

adv_neg  = -1.0
surr1_neg = r_range * adv_neg
surr2_neg = np.clip(r_range, 1-clip_eps, 1+clip_eps) * adv_neg
obj_neg   = np.minimum(surr1_neg, surr2_neg)

ax4.plot(r_range, surr1_pos, "--", color="#2ecc71", alpha=0.6, label="r·A (A>0)")
ax4.plot(r_range, surr2_pos, ":",  color="#2ecc71", alpha=0.6, label="clip(r)·A (A>0)")
ax4.plot(r_range, obj_pos,   "-",  color="#2ecc71", linewidth=2.5, label="min(…) A>0")

ax4.plot(r_range, surr1_neg, "--", color="#e74c3c", alpha=0.6, label="r·A (A<0)")
ax4.plot(r_range, surr2_neg, ":",  color="#e74c3c", alpha=0.6, label="clip(r)·A (A<0)")
ax4.plot(r_range, obj_neg,   "-",  color="#e74c3c", linewidth=2.5, label="min(…) A<0")

ax4.axvline(1-clip_eps, color="gray", linestyle="--", alpha=0.5)
ax4.axvline(1+clip_eps, color="gray", linestyle="--", alpha=0.5, label=f"clip bounds [1±{clip_eps}]")
ax4.axvline(1.0, color="black", linewidth=0.8, alpha=0.4)

# Scatter actual ratios
if len(ratios_np) > 0:
    ax4.hist(ratios_np, bins=40, density=True, alpha=0.15, color="#3498db",
             label=f"Actual ratio distribution\n(last 5 updates, n={len(ratios_np)})")

ax4.set_xlabel("Probability Ratio r = π_new / π_old")
ax4.set_ylabel("PPO Objective (per-sample)")
ax4.set_title("PPO-Clip Objective: Flat Regions = Clipped", fontweight="bold")
ax4.legend(fontsize=8, loc="upper left"); ax4.grid(alpha=0.3)
ax4.set_xlim(0.5, 1.8)

plt.suptitle("Chapter 9 — A2C vs PPO Results", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("ch09_results.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# Final performance summary
fig, ax = plt.subplots(figsize=(10, 4))
ax.axis("off")
envs = ["CartPole-v1", "LunarLander-v3"]
table_data = [
    ["Method", "CartPole (last 50)", "LunarLander (last 50)", "Epochs/rollout", "Key trick"],
    ["A2C",
     f"{np.mean(a2c_cartpole[-50:]):.0f} ± {np.std(a2c_cartpole[-50:]):.0f}",
     f"{np.mean(a2c_lunar[-50:]):.0f} ± {np.std(a2c_lunar[-50:]):.0f}",
     "1", "GAE + entropy bonus"],
    ["PPO-Clip",
     f"{np.mean(ppo_cartpole[-50:]):.0f} ± {np.std(ppo_cartpole[-50:]):.0f}",
     f"{np.mean(ppo_lunar[-50:]):.0f} ± {np.std(ppo_lunar[-50:]):.0f}",
     "4", "Ratio clipping ε=0.2"],
]
tbl = ax.table(cellText=table_data[1:], colLabels=table_data[0],
               cellLoc="center", loc="center",
               colColours=["#2c3e50"]*5)
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.scale(1.3, 1.9)
for (row, col), cell in tbl.get_celld().items():
    if row == 0:
        cell.set_text_props(color="white", fontweight="bold")
plt.title("Chapter 9 — Summary Table", fontsize=13, fontweight="bold", pad=20)
plt.tight_layout()
plt.savefig("ch09_summary.png", dpi=120, bbox_inches="tight")
plt.show()


<div style="background: #0d2137; padding: 20px; border-radius: 8px; border: 1px solid #1f6feb; margin-top: 20px;">
  <h3 style="color: #79c0ff; margin-top: 0;">Глава 9 — ключевые выводы</h3>
  <ul style="color: #c9d1d9; line-height: 1.8; margin: 12px 0 16px 0;">
    <li><strong>GAE ($\lambda$)</strong> при $\lambda \approx 0.95$ обычно даёт удобный компромисс смещение-дисперсия; значения, близкие к 1, помогают в начале обучения, когда функция ценности $V$ ещё неточна.</li>
    <li><strong>A2C</strong> — простая и стабильная базовая линия актёр-критик: собирает $n$-шаговый роллаут, использует GAE с общей сетью ценности и делает одно обновление на роллаут.</li>
    <li><strong>PPO-Clip</strong> переиспользует каждый $n$-шаговый роллаут для нескольких обновлений градиента, объединяет GAE с общей сетью ценности и улучшает сэмпл-эффективность за счёт отсечения отношения вероятностей.</li>
    <li><strong>Доля отсечения</strong> — полезный диагностический сигнал PPO: около 5–10% обычно норма, выше 30% указывает на слишком большие обновления (нужен меньший learning rate или меньшее <code>clip_eps</code>).</li>
    <li><strong>Ортогональная инициализация</strong> помогает удерживать начальную стратегию близкой к равномерной, что делает раннее исследование и оптимизацию более стабильными.</li>
  </ul>

  <div style="color: #c9d1d9; line-height: 1.8; margin: 0;">
    <strong style="color: #f0a500;">Сравнение алгоритмов:</strong>
    <ul style="margin: 8px 0 0 0; padding-left: 22px;">
      <li><strong>REINFORCE</strong> ждёт полного эпизода, затем делает одно обновление стратегии по Монте-Карло-отдачам, без replay и без сети ценности.</li>
      <li><strong>A2C</strong> переходит на $n$-шаговые роллауты, оценивает преимущества через GAE и использует общего критика, так что каждый роллаут даёт одно обновление с меньшей дисперсией.</li>
      <li><strong>PPO-Clip</strong> сохраняет структуру A2C/GAE, но переиспользует роллаут для нескольких эпох с отсечением отношения, ограничивающим смещение стратегии за одно обновление.</li>
    </ul>
  </div>
</div>

<div style="background: #1e3a5f; padding: 15px; border-radius: 8px; border-left: 4px solid #f0a500; margin-top: 20px;">
  <strong style="color: #f0a500;">Дальше:</strong> в главе 10 переходим к моделированию вознаграждения и его применениям.
</div>